In [1]:


import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [2]:
NUM_DIGITS = 5
NUM_CLASSES = 10
DIGIT_HEIGHT = 32
DIGIT_WIDTH = 32
EPOCHS = 50

MIN_COMPONENT_AREA = 0.02
MAX_COMPONENT_AREA = 0.60
BOX_PADDING_PIXELS = 2


In [3]:

# ========================= PATHS =========================
NOTEBOOK_DIR = os.path.abspath("")
BASE_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, "..", "data_set_generator_for_cnns_only", "data_set"))

TRAIN_IMG_DIR = os.path.join(BASE_DIR, "train")
VALID_IMG_DIR = os.path.join(BASE_DIR, "valid")
TEST_IMG_DIR = os.path.join(BASE_DIR, "test")

TRAIN_CSV = os.path.join(BASE_DIR, "train.csv")
VALID_CSV = os.path.join(BASE_DIR, "valid.csv")
TEST_CSV = os.path.join(BASE_DIR, "test.csv")


print("Checking paths...")
print(f"Train CSV exists: {os.path.exists(TRAIN_CSV)}")
print(f"Train Image Directory exists: {os.path.exists(TRAIN_IMG_DIR)}")
print(f"Valid CSV exists: {os.path.exists(VALID_CSV)}")
print(f"Valid Image Directory exists: {os.path.exists(VALID_IMG_DIR)}")
print(f"Test CSV exists: {os.path.exists(TEST_CSV)}")
print(f"Test Image Directory exists: {os.path.exists(TEST_IMG_DIR)}")

Checking paths...
Train CSV exists: True
Train Image Directory exists: True
Valid CSV exists: True
Valid Image Directory exists: True
Test CSV exists: True
Test Image Directory exists: True


In [5]:
# ========================= LOAD DATA (FAIL-SAFE) =========================
print("Loading datasets safely...")
df_train = pd.read_csv(TRAIN_CSV)
df_valid = pd.read_csv(VALID_CSV)
df_test = pd.read_csv(TEST_CSV)

for name, df, img_dir in [("train", df_train, TRAIN_IMG_DIR),
                           ("valid", df_valid, VALID_IMG_DIR),
                           ("test", df_test, TEST_IMG_DIR)]:
    mask = df['image'].apply(lambda x: os.path.exists(os.path.join(img_dir, str(x))))
    missing = len(df) - mask.sum()
    if missing > 0:
        print(f"Warning: {missing} rows in {name}.csv don't have matching files on disk. Skipping them.")

df_train = df_train[df_train['image'].apply(lambda x: os.path.exists(os.path.join(TRAIN_IMG_DIR, str(x))))]
df_valid = df_valid[df_valid['image'].apply(lambda x: os.path.exists(os.path.join(VALID_IMG_DIR, str(x))))]
df_test = df_test[df_test['image'].apply(lambda x: os.path.exists(os.path.join(TEST_IMG_DIR, str(x))))]

df_train['label'] = df_train['label'].astype(int)
df_valid['label'] = df_valid['label'].astype(int)
df_test['label'] = df_test['label'].astype(int)

print(f"\nFinal Ready Samples -> Train: {len(df_train)} | Val: {len(df_valid)} | Test: {len(df_test)}")


Loading datasets safely...

Final Ready Samples -> Train: 1270 | Val: 173 | Test: 212


In [6]:

# ========================= DIGIT SEGMENTATION =========================
def _binarize_for_segmentation(gray_img):
    """Denoise + threshold so digit strokes end up as white (255) on a
    black (0) background, regardless of the original polarity."""
    denoised = cv2.bilateralFilter(gray_img, d=5, sigmaColor=50, sigmaSpace=50)
    _, binary = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Otsu doesn't know which side is "foreground". Assume digit strokes are
    # the minority of pixels (true for most meter/ID crops); if white pixels
    # are the majority, the polarity is inverted, so flip it.
    if np.mean(binary == 255) > 0.5:
        binary = cv2.bitwise_not(binary)

    # Morphological close to bridge small gaps within a single digit's strokes.
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    return binary
